In [ ]:
import numpy as np  
import pandas as pd  
import seaborn as sns  
import warnings 

import matplotlib.pyplot as plt  

from time import time
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.stem import WordNetLemmatizer

warnings.filterwarnings('ignore') 

%matplotlib inline

In [2]:
# Загрузка данных
books = pd.read_csv('data/books.csv')
book_tags = pd.read_csv('data/book_tags.csv')
ratings = pd.read_csv('data/ratings.csv')
tags = pd.read_csv('data/tags.csv')

# <center> **Подготовка данных**

In [3]:
# Список числовых признаков
books_numeric = [x for x in books.columns if books[x].dtype != 'object']

# Проверим диапазон значений
for col in books_numeric:
    print(f'{col}: {books[col].min()} - {books[col].max()}')
    
    
# Преобразование типов данных
# uint16: books_count
# int16: original_publication_year
# uint32: остальные целочисленные, кроме идентификаторов
# int64: isbn13
# float16: average_rating
books['books_count'] = books['books_count'].astype('uint16')
books['original_publication_year'] = books['original_publication_year'].astype('Int16')
books['isbn13'] = books['isbn13'].astype('Int64')
books['average_rating'] = books['average_rating'].astype('float16')
for col in ['books_count', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5']:
    books[col] = books[col].astype('uint32')


id: 1 - 10000
book_id: 1 - 33288638
best_book_id: 1 - 35534230
work_id: 87 - 56399597
books_count: 1 - 3455
isbn13: 195170342.0 - 9790007672390.0
original_publication_year: -1750.0 - 2017.0
average_rating: 2.47 - 4.82
ratings_count: 2716 - 4780653
work_ratings_count: 5510 - 4942365
work_text_reviews_count: 3 - 155254
ratings_1: 11 - 456191
ratings_2: 30 - 436802
ratings_3: 323 - 793319
ratings_4: 750 - 1481305
ratings_5: 754 - 3011543


In [4]:
# book_tags['count'] -> uint32
book_tags['count'] = book_tags['count'].astype('uint32')
# ratings['rating'] -> uint8
ratings['rating'] = ratings['rating'].astype('uint8')

In [5]:
# Из books удалим признаки со ссылками на изображения
books = books.drop(['image_url', 'small_image_url'], axis=1)

In [6]:
# Проверка на наличие полных дубликатов
print(f'Количество дубликатов в books: {books.duplicated().sum()}')
print(f'Количество дубликатов в book_tags: {book_tags.duplicated().sum()}')
print(f'Количество дубликатов в tags: {tags.duplicated().sum()}')
print(f'Количество дубликатов в ratings: {ratings.duplicated().sum()}')

Количество дубликатов в books: 0
Количество дубликатов в book_tags: 6
Количество дубликатов в tags: 0
Количество дубликатов в ratings: 1644


In [7]:
# Дубликаты в book_tags
book_tags[book_tags.duplicated(keep=False)]

,goodreads_book_id,tag_id,count
159370,22369,25148,4
159371,22369,25148,4
265127,52629,10094,1
265128,52629,10094,1
265139,52629,2928,1
265140,52629,2928,1
265154,52629,13272,1
265155,52629,13272,1
265186,52629,13322,1
265187,52629,13322,1


In [8]:
print(f'Количество записей до удаления: {book_tags.shape[0]}')
# Удалим полные дубликаты в book_tags
book_tags.drop_duplicates(inplace=True)
print(f'Количество дубликатов в book_tags: {book_tags.duplicated().sum()}')
print(f'Количество записей после удаления: {book_tags.shape[0]}')

Количество записей до удаления: 999912
Количество дубликатов в book_tags: 0
Количество записей после удаления: 999906


In [9]:
# Дополнительно проверим дубликаты по признакам идентификаторов в ratings
print(f'Количество дубликатов в ratings по признакам book_id и user_id: {ratings.duplicated(subset=["book_id", "user_id"]).sum()}')

Количество дубликатов в ratings по признакам book_id и user_id: 2278


In [10]:
print(f'Количество записей до удаления: {ratings.shape[0]}')
# Удаление дубликатов, оставляя последнюю оценку
ratings = ratings.drop_duplicates(subset=['book_id', 'user_id'], keep='last')
print(f'Количество дубликатов в ratings: {ratings.duplicated().sum()}')
print(f'Количество записей после удаления: {ratings.shape[0]}')

Количество записей до удаления: 981756
Количество дубликатов в ratings: 0
Количество записей после удаления: 979478


In [11]:
# Удаляем пропуски в original_title
books = books.dropna(subset=['original_title'])

# <center> **Обучение моделей**

## Train / test разделение

+ Считаем сколько оценок поставил каждый пользователь
+ Отбираем только тех, у кого оценок больше 5 (остальные идут в выборку для холодного старта)
+ У каждого пользователя случайным образом выбираем 20% записей (test)
+ Остальные записи идут в train

In [ ]:
# Отбираем пользователей, у которых больше 5 оценок
users = ratings['user_id'].value_counts()
active_users = users[users > 5].index
print(len(active_users))

# Отбираем пользователей для холодного старта
cold_start_users = set(ratings['user_id'].unique()).difference(set(active_users))   
print(f'Количество пользователей в группе для холодного старта: {len(cold_start_users)}')
print(f'Количество активных пользователей: {len(active_users)}')

# Отфильтровываем датафреймы
hot_ratings = ratings[ratings['user_id'].isin(active_users)]
cold_ratings = ratings[ratings['user_id'].isin(cold_start_users)]

# Выбираем 20% записей в тест
def sample_test(group, test_size=0.2):
    n_users = max(1, int(len(group) * test_size))
    return group.sample(n=n_users)

test_groups = hot_ratings.groupby('user_id', group_keys=False).apply(sample_test)
test_idx = test_groups.index   
    
# Фильтруем датафреймы
test_df = hot_ratings.loc[test_idx]
train_df = hot_ratings.drop(test_idx) 

print(f'Размернось cold start: {cold_ratings.shape}')
print(f'Размернось train: {train_df.shape}')
print(f'Размернось test: {test_df.shape}')

35659
Количество пользователей в группе для холодного старта: 81
Количество активных пользователей: 35659


KeyboardInterrupt: 